# 梯度监测（核心）

| 指标名                 | 含义                | 用途            |
| :------------------ | :---------------- | :------------ |
| `actor/grad_norm`   | Actor 网络梯度 L2 范数  | **检测梯度消失/爆炸** |
| `critic/grad_norm`  | Critic 网络梯度 L2 范数 | 监控价值网络稳定性     |
| `grad_norm_clipped` | 裁剪后的梯度范数          | 验证裁剪是否生效      |
| `grad_norm_ratio`   | 裁剪后/裁剪前 比例        | 判断裁剪频率        |

## 特色功能：梯度熔断器
- 当单步梯度范数超过阈值（默认 1000）时，自动丢弃该 step 的梯度更新，防止污染模型
- 跨节点梯度一致性校验：聚合所有节点的梯度范数，检测分布式训练中的梯度分歧



# 四大梯度指标详解
## actor/grad_norm —— 策略网络的"脉搏"

### 计算原理：
- 对所有可训练参数求L2范数

In [ ]:
total_norm = 0.0
for p in actor_model.parameters():
    if p.grad is not None:
        total_norm += p.grad.data.norm(2).item()**2
grad_norm = total_norm ** 0.5

|     数值范围    | 状态      | 诊断               |
| :---------: | :------ | :--------------- |
|   `< 1e-4`  | 🚨 梯度消失 | 检查激活函数、奖励函数、学习率  |
| `1e-4 ~ 10` | ✅ 健康    | <font color='red'>正常训练区间  </font>         |
|  `10 ~ 100` | ⚠️ 偏高   | 可能学习率过大或奖励尺度不当   |
|   `> 1000`  | 🔥 梯度爆炸 | 触发熔断器，该 step 被丢弃 |


## critic/grad_norm —— 价值网络的"血压"
Critic（价值模型）负责估计状态/动作价值，其梯度稳定性直接影响优势估计的准确性。
- Critic 梯度 >> Actor 梯度：价值函数拟合困难，可能需要降低 Critic 学习率
- Critic 梯度 ≈ 0： <font color='red'>价值网络已收敛或陷入局部最优，GAE 估计将失效</font>

## grad_norm_clipped —— 裁剪后的"实际步长"

In [ ]:
# 框架内部逻辑
grad_norm_before = compute_grad_norm()
torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)  # 裁剪
grad_norm_after = compute_grad_norm()

| 对比                  | 含义           |
| :------------------ | :----------- |
| `clipped ≈ before`  | 梯度正常，未触发裁剪   |
| `clipped << before` | 梯度爆炸被压制，裁剪生效 |


## grad_norm_ratio —— 裁剪频率的"晴雨表"

$ratio = \frac{grad_norm_clipped}{grad_norm_before}$

|  ratio 范围  | 含义                      |
| :--------: | :---------------------- |
|   `≈ 1.0`  | 该 step 未触发裁剪，梯度健康       |
|   `< 0.5`  | <font color='red'>频繁触发裁剪，说明梯度持续偏大  </font>       |
| 持续 `< 0.1` | 训练处于不稳定状态，需降低学习率或检查奖励函数 |


# 梯度熔断器（Gradient Circuit Breaker）

## 为什么需要熔断器？
在 LLM+RL 中，以下情况可能导致单步梯度爆炸：
- 数值异常：某个 token 的 importance ratio 因旧策略概率接近 0 而趋于无穷
- <font color='red'>奖励尖峰：某个样本获得极端奖励（如 +100），反向传播时梯度放大</font>
- 分布式不一致：某节点因硬件/通信问题产生脏梯度

如果没有熔断器，一次异常更新可能：
- 永久破坏 LayerNorm 的统计量
- 将 Embedding 层某个 token 的权重推到荒谬值
- 导致后续所有 step 的 KL 散度失控

## 核心机制

In [ ]:
GRAD_NORM_BREAKER = 1000.0 # 默认阀值

grad_norm = compute_grad_norm(actor_model)

if grad_norm > GRAD_NORM_BREAKER: # 超过阀值
    # 1. 丢弃该step的梯度
    optimizer.zero_grad()
    # 2. 记录异常值
    log.warning(f"Step {step}: grad_norm={grad_norm:.2f} > {GRAD_NORM_BREAKER},dropping update!")
    # 3. 可选：保存诊断快照
    save_debug_snapshot(step,loss,logits,rewards)
    # 4. 跳过optimizer.step()
    return # 不执行更新

# 跨节点梯度一致性校验

在分布式训练中，<font color='red'>各节点的梯度理论上应该一致（因为数据并行时梯度会 all-reduce 平均）</font>。校验逻辑：


In [ ]:
# 收集所以后节点的actor grad_norm

local_norm = compute_grad_norm(actor_model)
all_norms = all_gather(local_norm) # [world_size]

max_diff = max(all_norms) - min(all_norms)

if max_diff > 1e-5:
    log.error(f"Gradient divergence detected! max_diff={max_diff:.2e}")

# 完整使用示例
场景：基于 verl 风格的训练监控脚本

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict
import math

class GradientMonitor:
    """
    verl 风格的梯度监测器
    支持：grad_norm、clipped_norm、ratio、熔断器、分布式校验
    """
    def __init__(self, actor_model, critic_model=None, 
                 grad_clip=1.0, grad_breaker=1000.0, 
                 world_size=1, rank=0):
        self.actor = actor_model
        self.critic = critic_model
        self.grad_clip = grad_clip
        self.grad_breaker = grad_breaker
        self.world_size = world_size
        self.rank = rank
        
        self.dropped_steps = 0
        self.history = defaultdict(list)
    
    def compute_grad_norm(self, model):
        """计算模型所有参数的 L2 范数"""
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2).item()
                total_norm += param_norm ** 2
        return math.sqrt(total_norm)
    
    def check_distributed_consistency(self, grad_norm):
        """跨节点梯度一致性校验（模拟）"""
        if self.world_size == 1:
            return 0.0, True
        
        # 实际场景中：torch.distributed.all_gather/all_reduce
        # 这里模拟多节点差异
        all_norms = [grad_norm * (1 + 0.01 * i) for i in range(self.world_size)]
        max_diff = max(all_norms) - min(all_norms)
        is_consistent = max_diff < 1e-5
        return max_diff, is_consistent
    
    def step(self, step_id, actor_optimizer, critic_optimizer=None):
        """
        在 optimizer.step() 之前调用
        返回：是否继续执行更新（False = 熔断）
        """
        # 1. 计算裁剪前范数
        actor_norm_before = self.compute_grad_norm(self.actor)
        actor_norm_before_log = actor_norm_before
        
        # 2. 梯度熔断检查
        if actor_norm_before > self.grad_breaker:
            self.dropped_steps += 1
            # 清空梯度，跳过更新
            actor_optimizer.zero_grad()
            if critic_optimizer:
                critic_optimizer.zero_grad()
            
            print(f"🚨 [Step {step_id}] GRADIENT CIRCUIT BREAKER TRIGGERED!")
            print(f"   grad_norm={actor_norm_before:.2f} > breaker={self.grad_breaker}")
            print(f"   Total dropped steps: {self.dropped_steps}")
            
            self.history['breaker_triggered'].append(step_id)
            return False  # 熔断：不执行 step
        
        # 3. 执行裁剪（Actor）
        torch.nn.utils.clip_grad_norm_(self.actor.parameters(), self.grad_clip)
        actor_norm_after = self.compute_grad_norm(self.actor)
        
        # 4. Critic 裁剪（如有）
        critic_norm_before, critic_norm_after = 0.0, 0.0
        if self.critic:
            critic_norm_before = self.compute_grad_norm(self.critic)
            torch.nn.utils.clip_grad_norm_(self.critic.parameters(), self.grad_clip)
            critic_norm_after = self.compute_grad_norm(self.critic)
        
        # 5. 计算 Ratio
        ratio = actor_norm_after / (actor_norm_before + 1e-8)
        
        # 6. 分布式一致性
        max_diff, consistent = self.check_distributed_consistency(actor_norm_before)
        
        # 7. 记录历史
        self.history['actor_norm_before'].append(actor_norm_before)
        self.history['actor_norm_after'].append(actor_norm_after)
        self.history['critic_norm'].append(critic_norm_after if self.critic else 0)
        self.history['ratio'].append(ratio)
        self.history['distributed_diff'].append(max_diff)
        
        # 8. 打印日志
        if step_id % 10 == 0:
            status = "✅" if actor_norm_before < self.grad_clip else "⚠️ CLIPPED"
            print(f"[Step {step_id:4d}] {status} "
                  f"Actor: {actor_norm_before:.4f} → {actor_norm_after:.4f} "
                  f"(ratio={ratio:.3f}) | "
                  f"Critic: {critic_norm_after:.4f} | "
                  f"DistDiff: {max_diff:.2e}")
            
            # 异常警告
            if actor_norm_before < 1e-4:
                print(f"   ⚠️  Warning: Gradient vanishing detected!")
            if not consistent:
                print(f"   ⚠️  Warning: Distributed gradient divergence!")
        
        return True  # 正常：继续执行 step


# ==================== 模拟训练演示 ====================

class DummyActor(nn.Module):
    def __init__(self, vocab=1000, dim=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.fc = nn.Linear(dim, vocab)
    
    def forward(self, x):
        return self.fc(self.embed(x))

class DummyCritic(nn.Module):
    def __init__(self, vocab=1000, dim=256):
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.fc = nn.Linear(dim, 1)
    
    def forward(self, x):
        return self.fc(self.embed(x))

# 初始化
actor = DummyActor()
critic = DummyCritic()
actor_opt = torch.optim.Adam(actor.parameters(), lr=1e-4)
critic_opt = torch.optim.Adam(critic.parameters(), lr=1e-4)

monitor = GradientMonitor(
    actor_model=actor,
    critic_model=critic,
    grad_clip=1.0,        # 裁剪阈值
    grad_breaker=100.0,   # 熔断阈值（设低一点方便演示）
)

print("=" * 70)
print("开始模拟 RL 训练...")
print("=" * 70)

for step in range(50):
    # 模拟前向
    batch_size, seq_len = 4, 10
    input_ids = torch.randint(0, 1000, (batch_size, seq_len))
    
    logits = actor(input_ids)
    values = critic(input_ids)
    
    # 模拟损失
    loss_actor = -logits.mean() * torch.randn(1)  # 带噪声
    loss_critic = F.mse_loss(values, torch.randn_like(values))
    
    # 反向传播
    actor_opt.zero_grad()
    critic_opt.zero_grad()
    loss_actor.backward()
    loss_critic.backward()
    
    # 模拟异常：第 25 步制造梯度爆炸
    if step == 25:
        for p in actor.parameters():
            if p.grad is not None:
                p.grad *= 500.0  # 人为制造爆炸
        print(f"\n💥 [Step {step}] Injecting artificial gradient explosion!\n")
    
    # 监测 + 熔断判断
    should_update = monitor.step(step, actor_opt, critic_opt)
    
    if should_update:
        actor_opt.step()
        critic_opt.step()
    # else: 已被熔断，跳过 step

print("\n" + "=" * 70)
print("训练结束，统计摘要：")
print(f"  总步数: 50")
print(f"  熔断步数: {monitor.dropped_steps}")
print(f"  Actor 梯度范围: [{min(monitor.history['actor_norm_before']):.2e}, {max(monitor.history['actor_norm_before']):.2f}]")
print(f"  平均裁剪比例: {sum(monitor.history['ratio'])/len(monitor.history['ratio']):.3f}")
print("=" * 70)

# 输出示例解读

In [ ]:
[Step    0] ✅ Actor: 0.0234 → 0.0234 (ratio=1.000) | Critic: 0.0156 | DistDiff: 0.00e+00
[Step   10] ✅ Actor: 0.0456 → 0.0456 (ratio=1.000) | Critic: 0.0321 | DistDiff: 0.00e+00
[Step   20] ⚠️ CLIPPED Actor: 2.3456 → 1.0000 (ratio=0.426) | Critic: 0.0891 | DistDiff: 0.00e+00

💥 [Step 25] Injecting artificial gradient explosion!

🚨 [Step 25] GRADIENT CIRCUIT BREAKER TRIGGERED!
   grad_norm=5234.56 > breaker=100.0
   Total dropped steps: 1

[Step   30] ✅ Actor: 0.0567 → 0.0567 (ratio=1.000) | Critic: 0.0412 | DistDiff: 0.00e+00

# 生产环境配置建议

In [ ]:
# verl 配置示例
trainer:
  grad_norm_clip: 1.0          # 常规裁剪阈值
  grad_norm_breaker: 1000.0    # 熔断阈值（设为 clip 的 1000 倍）
  
  # 分布式校验
  check_distributed_grad: true
  grad_divergence_tolerance: 1e-5
  
  # 日志
  log_interval: 10
  log_grad_norm: true
  log_grad_ratio: true
  log_critic_grad: true

# 一句话总结
verl 的梯度监测是"四层防护"：先测范数看健康度，再裁剪控步长，再算 ratio 看频率，最后熔断保底线。跨节点一致性校验则是分布式训练的"心电图"——任何节点的异常梯度都无处遁形。